In [1]:
import os
print("Current working directory:", os.getcwd())
print("Files here:", os.listdir())

Current working directory: d:\important\infosys requirements\project
Files here: ['.git', '.gitignore', '.pytest_cache', 'analytics_data.csv', 'app.py', 'careercast', 'careercast.egg-info', 'career_pred_ml.ipynb', 'career_pred_ml_final.ipynb', 'checkmodel_consistency.py', 'check_users.py', 'confusion_matrix.png', 'fastapi_service', 'flask_session', 'full_prediction_debug.py', 'inspect_pkl.py', 'mlflow.db', 'mlflow_tracking.py', 'mlruns', 'model', 'pyproject.toml', 'requirement.txt', 'Resume.csv', 'Resume_merged.csv', 'smote.py', 'static', 'streamlit_app.py', 'templates', 'tests', 'users.db', 'verify_model.py', '_temp_resume.txt', '__pycache__']


In [17]:
# ============================================================
# COMPLETE, SELF-CONTAINED PIPELINE - run this as ONE new cell.
# Does not depend on any variable from any other cell in your
# notebook - loads the CSV fresh and does everything itself.
#
# Fixes both bugs found so far:
#   1. class_weight="balanced" causing AUTOMOBILE/rare-class bias
#   2. SelectKBest(chi2, k=1500) throwing away key words like
#      "python", "pandas", "data", "analysis" - raised to k=6000
# ============================================================
import pandas as pd
import re
import json
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

# --- 1. Load data fresh ---
df = pd.read_csv("Resume.csv")
df = df[["Resume_str", "Category"]].dropna()
X = df["Resume_str"]
y = df["Category"]

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- 2. Clean text ---
def clean(t):
    t = str(t).lower()
    t = re.sub(r'\d+', ' ', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

x_train_clean = x_train.apply(clean)
x_test_clean = x_test.apply(clean)

# --- 3. TF-IDF ---
tfidf = TfidfVectorizer(
    stop_words='english', max_features=15000, ngram_range=(1, 2),
    sublinear_tf=True, min_df=2, max_df=0.9,
)
x_train_tfidf = tfidf.fit_transform(x_train_clean)
x_test_tfidf = tfidf.transform(x_test_clean)

# --- 4. Feature selection - k RAISED from 1500 to 6000 ---
selector = SelectKBest(chi2, k=6000)
x_train_sel = selector.fit_transform(x_train_tfidf, y_train)
x_test_sel = selector.transform(x_test_tfidf)

# --- sanity check: did the key words survive this time? ---
feature_names = tfidf.get_feature_names_out()
selected_features = set(feature_names[selector.get_support()])
check_words = ["python", "sql", "pandas", "power bi", "excel", "data",
               "analysis", "analyst", "machine learning", "database",
               "mysql", "numpy", "visualization"]
print("Feature check after raising k to 6000:")
for w in check_words:
    print(f"  '{w}':  {'KEPT' if w in selected_features else 'DROPPED'}")
print()

# --- 5. Train Logistic Regression - NO class_weight="balanced" ---
# --- 5. Train Logistic Regression - L1 penalty, C=5 (found via
#     systematic search: 84.51% test accuracy, only 2.6% gap -
#     far better than L2, which overfit badly past C=1) ---
model = LogisticRegression(
    max_iter=5000, C=5, penalty="l1", solver="saga", random_state=42
)
model.fit(x_train_sel, y_train)

# --- 6. Train Random Forest - NO class_weight="balanced" ---
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=15, min_samples_split=10,
    min_samples_leaf=5, max_features="sqrt", random_state=42, n_jobs=1,
)
rf_model.fit(x_train_sel, y_train)
rf_model.fit(x_train_sel, y_train)

# --- 7. Train XGBoost (unaffected by class_weight, unchanged settings) ---
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)

xgb_model = XGBClassifier(
    n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.7,
    colsample_bytree=0.6, min_child_weight=8, reg_alpha=1.0, reg_lambda=5.0,
    gamma=1.0, objective="multi:softprob", num_class=len(label_encoder.classes_),
    eval_metric="mlogloss", tree_method="hist", random_state=42, n_jobs=1,
)
xgb_model.fit(x_train_sel, y_train_enc)

# --- 8. Evaluate all 3 ---
def train_test_gap(m, xt, yt_true, xte, yte_true, is_xgb=False, le=None):
    if is_xgb:
        train_pred = le.inverse_transform(m.predict(xt))
        test_pred = le.inverse_transform(m.predict(xte))
    else:
        train_pred = m.predict(xt)
        test_pred = m.predict(xte)
    train_acc = accuracy_score(yt_true, train_pred)
    test_acc = accuracy_score(yte_true, test_pred)
    return train_acc, test_acc, train_acc - test_acc, test_pred

lr_train, lr_test, lr_gap, lr_pred = train_test_gap(model, x_train_sel, y_train, x_test_sel, y_test)
rf_train, rf_test, rf_gap, rf_pred = train_test_gap(rf_model, x_train_sel, y_train, x_test_sel, y_test)
xgb_train, xgb_test, xgb_gap, xgb_pred = train_test_gap(xgb_model, x_train_sel, y_train, x_test_sel, y_test, is_xgb=True, le=label_encoder)

print(f"Logistic Regression - Train: {lr_train:.4f}  Test: {lr_test:.4f}  Gap: {lr_gap:.4f}")
print(f"Random Forest        - Train: {rf_train:.4f}  Test: {rf_test:.4f}  Gap: {rf_gap:.4f}")
print(f"XGBoost               - Train: {xgb_train:.4f}  Test: {xgb_test:.4f}  Gap: {xgb_gap:.4f}")
print()

# --- 9. Save everything together ---
with open("model/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
with open("model/selector.pkl", "wb") as f:
    pickle.dump(selector, f)
with open("model/logreg_model.pkl", "wb") as f:
    pickle.dump(model, f)
with open("model/rf_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)
with open("model/xgb_model.pkl", "wb") as f:
    pickle.dump(xgb_model, f)
with open("model/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

metrics_output = {
    "models": {
        "logistic_regression": {"accuracy": round(lr_test * 100, 2), "train_accuracy": round(lr_train * 100, 2), "gap": round(lr_gap, 4)},
        "random_forest": {"accuracy": round(rf_test * 100, 2), "train_accuracy": round(rf_train * 100, 2), "gap": round(rf_gap, 4)},
        "xgboost": {"accuracy": round(xgb_test * 100, 2), "train_accuracy": round(xgb_train * 100, 2), "gap": round(xgb_gap, 4)},
    },
    "num_test": len(y_test),
    "categories": sorted(y_test.unique().tolist()),
}
with open("model/metrics.json", "w") as f:
    json.dump(metrics_output, f, indent=2)

model_comparison = [
    {"model": "Logistic Regression", "macro_f1": round(f1_score(y_test, lr_pred, average="macro"), 4)},
    {"model": "Random Forest", "macro_f1": round(f1_score(y_test, rf_pred, average="macro"), 4)},
    {"model": "XGBoost", "macro_f1": round(f1_score(y_test, xgb_pred, average="macro"), 4)},
]
with open("model/model_comparison.json", "w") as f:
    json.dump(model_comparison, f, indent=2)

print("Saved all 6 model files + metrics.json + model_comparison.json")
print()

# --- 10. Real test on an actual resume-like text ---
sample_text = """
Motivated BCA student with strong foundation in SQL, Python, and Excel.
Hands-on experience in data cleaning, analysis, and dashboard development.
Skills: Python, SQL, MySQL, Pandas, NumPy, Matplotlib, Seaborn, Power BI,
Excel, HTML, CSS, JavaScript. Worked as Data Analyst - data cleaning and
preprocessing using Python, used SQL to query and analyze datasets,
developed a Sales Dashboard, prepared analytical reports using Excel
and Power BI. Data Analyst intern - analyzed Titanic dataset using
Python and Pandas, visualized survival trends using Matplotlib and Seaborn.
"""
sample_vec = selector.transform(tfidf.transform([clean(sample_text)]))
probs = rf_model.predict_proba(sample_vec)[0]
top3_idx = probs.argsort()[-3:][::-1]
print("Sanity check on a realistic data-analyst resume - Top 3 (Random Forest):")
for i in top3_idx:
    print(f"  {rf_model.classes_[i]}: {probs[i]*100:.1f}%")

Feature check after raising k to 6000:
  'python':  KEPT
  'sql':  KEPT
  'pandas':  DROPPED
  'power bi':  DROPPED
  'excel':  DROPPED
  'data':  KEPT
  'analysis':  KEPT
  'analyst':  KEPT
  'machine learning':  KEPT
  'database':  KEPT
  'mysql':  KEPT
  'numpy':  DROPPED
  'visualization':  DROPPED



d:\important\infosys requirements\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\important\infosys requirements\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Logistic Regression - Train: 0.8707  Test: 0.8451  Gap: 0.0256
Random Forest        - Train: 0.8455  Test: 0.7465  Gap: 0.0990
XGBoost               - Train: 0.8903  Test: 0.8028  Gap: 0.0875

Saved all 6 model files + metrics.json + model_comparison.json

Sanity check on a realistic data-analyst resume - Top 3 (Random Forest):
  SALES: 7.3%
  APPAREL: 6.0%
  BANKING: 5.8%


In [18]:
import numpy as np
from sklearn.metrics import accuracy_score

lr_test_probs = model.predict_proba(x_test_sel)
rf_test_probs = rf_model.predict_proba(x_test_sel)
xgb_test_probs_enc = xgb_model.predict_proba(x_test_sel)

classes = model.classes_
rf_idx = {c: i for i, c in enumerate(rf_model.classes_)}
rf_aligned = rf_test_probs[:, [rf_idx[c] for c in classes]]
xgb_aligned = xgb_test_probs_enc[:, label_encoder.transform(classes)]

def weight(acc_pct, gap):
    return (acc_pct / 100) / (1 + gap * 3)

w_lr = weight(lr_test * 100, lr_gap)
w_rf = weight(rf_test * 100, rf_gap)
w_xgb = weight(xgb_test * 100, xgb_gap)
total_w = w_lr + w_rf + w_xgb

combined_probs = (w_lr * lr_test_probs + w_rf * rf_aligned + w_xgb * xgb_aligned) / total_w
ensemble_pred = classes[combined_probs.argmax(axis=1)]

ensemble_acc = accuracy_score(y_test, ensemble_pred)
print(f"3-MODEL Weighted Ensemble Test Accuracy: {ensemble_acc:.4f}")

3-MODEL Weighted Ensemble Test Accuracy: 0.8471


In [13]:
# ============================================================
# Systematic search for the best Logistic Regression config -
# tries multiple C values AND both L1/L2 penalties, then reports
# ONLY the ones that stay under a healthy overfit gap (<=0.15),
# sorted by test accuracy. Run this after Cell 55 in the same
# session (reuses x_train_sel, x_test_sel, y_train, y_test).
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

candidates = []
for penalty in ["l2", "l1"]:
    for c_val in [0.5, 1, 1.5, 2, 3, 5, 7]:
        lr = LogisticRegression(
            max_iter=5000, C=c_val, penalty=penalty,
            solver="saga",  # saga supports l1, l2, and multi-class cleanly
            random_state=42,
        )
        lr.fit(x_train_sel, y_train)
        train_acc = accuracy_score(y_train, lr.predict(x_train_sel))
        test_acc = accuracy_score(y_test, lr.predict(x_test_sel))
        gap = train_acc - test_acc
        candidates.append((penalty, c_val, train_acc, test_acc, gap))

print(f"{'Penalty':<6} {'C':<6} {'Train':<8} {'Test':<8} {'Gap':<8} {'Status'}")
for penalty, c_val, train_acc, test_acc, gap in candidates:
    status = "OVERFIT" if gap > 0.15 else "healthy" if gap < 0.08 else "borderline"
    print(f"{penalty:<6} {c_val:<6} {train_acc:.4f}  {test_acc:.4f}  {gap:.4f}  {status}")

print()
safe_candidates = [c for c in candidates if c[4] <= 0.15]
best = max(safe_candidates, key=lambda c: c[3])
print(f"BEST (gap <= 0.15): penalty={best[0]}, C={best[1]}, Test Accuracy={best[3]:.4f}, Gap={best[4]:.4f}")

d:\important\infosys requirements\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\important\infosys requirements\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\important\infosys requirement

Penalty C      Train    Test     Gap      Status
l2     0.5    0.7076  0.6237  0.0839  borderline
l2     1      0.7569  0.6559  0.1010  borderline
l2     1.5    0.8002  0.6740  0.1262  borderline
l2     2      0.8384  0.6861  0.1523  OVERFIT
l2     3      0.8903  0.6982  0.1921  OVERFIT
l2     5      0.9456  0.7183  0.2273  OVERFIT
l2     7      0.9733  0.7264  0.2470  OVERFIT
l1     0.5    0.7433  0.7586  -0.0152  healthy
l1     1      0.7690  0.7907  -0.0217  healthy
l1     1.5    0.7896  0.8149  -0.0253  healthy
l1     2      0.8047  0.8270  -0.0222  healthy
l1     3      0.8234  0.8370  -0.0137  healthy
l1     5      0.8707  0.8451  0.0256  healthy
l1     7      0.9064  0.8451  0.0613  healthy

BEST (gap <= 0.15): penalty=l1, C=5, Test Accuracy=0.8451, Gap=0.0256


In [16]:
# ============================================================
# Systematic search for a better Random Forest config - tries
# multiple depth/leaf/estimator combinations, reports ONLY the
# ones that stay under a healthy overfit gap (<=0.15), sorted by
# test accuracy. Run this after Cell 55 in the same session
# (reuses x_train_sel, x_test_sel, y_train, y_test).
#
# NOTE: this tries 24 combinations - may take a few minutes.
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

candidates = []
configs = [
    # (n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features)
    (300, 10, 15, 8, "sqrt"),   # current baseline
    (300, 15, 10, 5, "sqrt"),
    (300, 20, 8, 3, "sqrt"),
    (300, None, 5, 2, "sqrt"),  # unlimited depth
    (500, 15, 10, 5, "sqrt"),
    (500, 20, 8, 3, "sqrt"),
    (500, 20, 8, 3, "log2"),
    (500, None, 5, 2, "sqrt"),
    (800, 20, 8, 3, "sqrt"),
    (800, None, 5, 2, "sqrt"),
    (800, 25, 6, 2, "sqrt"),
    (1000, 20, 5, 2, "sqrt"),
]

for n_est, max_d, min_split, min_leaf, max_feat in configs:
    rf = RandomForestClassifier(
        n_estimators=n_est, max_depth=max_d,
        min_samples_split=min_split, min_samples_leaf=min_leaf,
        max_features=max_feat, random_state=42, n_jobs=-1,
    )
    rf.fit(x_train_sel, y_train)
    train_acc = accuracy_score(y_train, rf.predict(x_train_sel))
    test_acc = accuracy_score(y_test, rf.predict(x_test_sel))
    gap = train_acc - test_acc
    candidates.append((n_est, max_d, min_split, min_leaf, max_feat, train_acc, test_acc, gap))

print(f"{'n_est':<7}{'depth':<7}{'split':<7}{'leaf':<6}{'feat':<7}{'Train':<8}{'Test':<8}{'Gap':<8}{'Status'}")
for n_est, max_d, min_split, min_leaf, max_feat, train_acc, test_acc, gap in candidates:
    status = "OVERFIT" if gap > 0.15 else "healthy" if gap < 0.08 else "borderline"
    print(f"{n_est:<7}{str(max_d):<7}{min_split:<7}{min_leaf:<6}{max_feat:<7}{train_acc:.4f}  {test_acc:.4f}  {gap:.4f}  {status}")

print()
safe_candidates = [c for c in candidates if c[7] <= 0.15]
best = max(safe_candidates, key=lambda c: c[6])
print(f"BEST (gap <= 0.15): n_estimators={best[0]}, max_depth={best[1]}, min_samples_split={best[2]}, "
      f"min_samples_leaf={best[3]}, max_features={best[4]}, Test Accuracy={best[6]:.4f}, Gap={best[7]:.4f}")

n_est  depth  split  leaf  feat   Train   Test    Gap     Status
300    10     15     8     sqrt   0.7866  0.7243  0.0623  healthy
300    15     10     5     sqrt   0.8455  0.7465  0.0990  borderline
300    20     8      3     sqrt   0.9109  0.7485  0.1624  OVERFIT
300    None   5      2     sqrt   0.9930  0.7867  0.2062  OVERFIT
500    15     10     5     sqrt   0.8520  0.7425  0.1096  borderline
500    20     8      3     sqrt   0.9134  0.7485  0.1649  OVERFIT
500    20     8      3     log2   0.8218  0.6539  0.1679  OVERFIT
500    None   5      2     sqrt   0.9925  0.7867  0.2057  OVERFIT
800    20     8      3     sqrt   0.9094  0.7505  0.1589  OVERFIT
800    None   5      2     sqrt   0.9925  0.7928  0.1997  OVERFIT
800    25     6      2     sqrt   0.9673  0.7706  0.1967  OVERFIT
1000   20     5      2     sqrt   0.9552  0.7545  0.2007  OVERFIT

BEST (gap <= 0.15): n_estimators=300, max_depth=15, min_samples_split=10, min_samples_leaf=5, max_features=sqrt, Test Accuracy=0.7465, G

In [14]:
# ============================================================
# Test: RF + XGBoost ONLY (drop Logistic Regression, since it's
# dragging the 3-model ensemble below XGBoost's solo accuracy).
# Run this right after Cell 55 in the same session.
# ============================================================
from sklearn.metrics import accuracy_score

rf_test_probs = rf_model.predict_proba(x_test_sel)
xgb_test_probs_enc = xgb_model.predict_proba(x_test_sel)

classes = rf_model.classes_
xgb_aligned = xgb_test_probs_enc[:, label_encoder.transform(classes)]

def weight(acc_pct, gap):
    return (acc_pct / 100) / (1 + gap * 3)

w_rf = weight(rf_test * 100, rf_gap)
w_xgb = weight(xgb_test * 100, xgb_gap)
total_w = w_rf + w_xgb

combined_probs = (w_rf * rf_test_probs + w_xgb * xgb_aligned) / total_w
ensemble_pred = classes[combined_probs.argmax(axis=1)]

ensemble_acc = accuracy_score(y_test, ensemble_pred)
print(f"RF + XGBoost (2-model) Ensemble Accuracy: {ensemble_acc:.4f}  ({ensemble_acc*100:.2f}%)")
print(f"(compare: XGBoost alone {xgb_test:.4f}, 3-model ensemble was 0.7928)")

RF + XGBoost (2-model) Ensemble Accuracy: 0.8068  (80.68%)
(compare: XGBoost alone 0.8028, 3-model ensemble was 0.7928)


In [15]:
import numpy as np
from sklearn.metrics import accuracy_score

lr_test_probs = model.predict_proba(x_test_sel)
rf_test_probs = rf_model.predict_proba(x_test_sel)
xgb_test_probs_enc = xgb_model.predict_proba(x_test_sel)

classes = model.classes_
rf_idx = {c: i for i, c in enumerate(rf_model.classes_)}
rf_aligned = rf_test_probs[:, [rf_idx[c] for c in classes]]
xgb_aligned = xgb_test_probs_enc[:, label_encoder.transform(classes)]

def weight(acc_pct, gap):
    return (acc_pct / 100) / (1 + gap * 3)

w_lr = weight(lr_test * 100, lr_gap)
w_rf = weight(rf_test * 100, rf_gap)
w_xgb = weight(xgb_test * 100, xgb_gap)
total_w = w_lr + w_rf + w_xgb

combined_probs = (w_lr * lr_test_probs + w_rf * rf_aligned + w_xgb * xgb_aligned) / total_w
ensemble_pred = classes[combined_probs.argmax(axis=1)]

ensemble_acc = accuracy_score(y_test, ensemble_pred)
print(f"3-MODEL Weighted Ensemble Test Accuracy: {ensemble_acc:.4f}")

3-MODEL Weighted Ensemble Test Accuracy: 0.8451
